# 01 - Data Preparation (OASIS-2)

Ce notebook extrait la partie preparation des donnees du notebook original.
Il prepare les splits train/test/shadow et sauvegarde les donnees pour la suite.

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
FEATURE_COLS = ["MR Delay", "Age", "EDUC", "SES", "MMSE", "CDR", "eTIV", "nWBV", "ASF"]
TARGET_COL = "Group"
TEST_SIZE = 0.80

data_candidates = ["oasis2.csv", "oasis_longitudinal.csv"]
data_path = None
for p in data_candidates:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError("Aucun fichier trouve: oasis2.csv ou oasis_longitudinal.csv")

print("=== Chargement des donnees ===")
print(f"Fichier utilise: {data_path}")
df = pd.read_csv(data_path)
print(f"Shape brute: {df.shape}")

=== Chargement des donnees ===
Fichier utilise: oasis_longitudinal.csv
Shape brute: (373, 15)


In [2]:
print("\n=== Nettoyage et encodage cible ===")
print(df[FEATURE_COLS + [TARGET_COL]].isna().sum())

df = df.dropna(subset=FEATURE_COLS + [TARGET_COL]).copy()
df[TARGET_COL] = (df[TARGET_COL] == "Demented").astype(int)

X = df[FEATURE_COLS].values.astype(np.float32)
y = df[TARGET_COL].values.astype(np.int32)

print("Shape nettoyee:", X.shape)
print(f"Classe 0: {(y == 0).sum()} | Classe 1: {(y == 1).sum()}")
print(f"Ratio classe 1: {y.mean():.4f}")

print("\n=== Split baseline train/test ===")
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)
print(f"Train size: {len(X_train_raw)} | Test size: {len(X_test_raw)}")
print(f"Train ratio classe 1: {y_train.mean():.4f}")
print(f"Test ratio classe 1: {y_test.mean():.4f}")

print("\n=== Split shadow pool (pour attaques shadow) ===")
X_ab, X_shadow_raw, y_ab, y_shadow = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
print(f"Shadow pool size: {len(X_shadow_raw)}")
print(f"Shadow ratio classe 1: {y_shadow.mean():.4f}")


=== Nettoyage et encodage cible ===
MR Delay     0
Age          0
EDUC         0
SES         19
MMSE         2
CDR          0
eTIV         0
nWBV         0
ASF          0
Group        0
dtype: int64
Shape nettoyee: (354, 9)
Classe 0: 227 | Classe 1: 127
Ratio classe 1: 0.3588

=== Split baseline train/test ===
Train size: 70 | Test size: 284
Train ratio classe 1: 0.3571
Test ratio classe 1: 0.3592

=== Split shadow pool (pour attaques shadow) ===
Shadow pool size: 107
Shadow ratio classe 1: 0.3551


In [3]:
print("\n=== Normalisation des features (baseline train/test) ===")
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_test = scaler.transform(X_test_raw).astype(np.float32)

X_train_seq = X_train.reshape(-1, 1, X_train.shape[1])
X_test_seq = X_test.reshape(-1, 1, X_test.shape[1])

print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")
print(f"X_train_seq shape: {X_train_seq.shape} | X_test_seq shape: {X_test_seq.shape}")


=== Normalisation des features (baseline train/test) ===
X_train shape: (70, 9) | X_test shape: (284, 9)
X_train_seq shape: (70, 1, 9) | X_test_seq shape: (284, 1, 9)


In [4]:
output_path = "prepared_oasis2_transformer.npz"

np.savez_compressed(
    output_path,
    X=X,
    y=y,
    X_train_raw=X_train_raw,
    X_test_raw=X_test_raw,
    y_train=y_train,
    y_test=y_test,
    X_train=X_train,
    X_test=X_test,
    X_train_seq=X_train_seq,
    X_test_seq=X_test_seq,
    X_shadow_raw=X_shadow_raw,
    y_shadow=y_shadow,
    scaler_mean=scaler.mean_.astype(np.float32),
    scaler_scale=scaler.scale_.astype(np.float32),
    feature_cols=np.array(FEATURE_COLS, dtype=object),
    target_col=np.array([TARGET_COL], dtype=object),
    seed=np.array([SEED], dtype=np.int32)
)

print("=== Sauvegarde terminee ===")
print(f"Fichier cree: {output_path}")
loaded = np.load(output_path, allow_pickle=True)
print("Contenu sauvegarde:")
print(sorted(list(loaded.files)))

=== Sauvegarde terminee ===
Fichier cree: prepared_oasis2_transformer.npz
Contenu sauvegarde:
['X', 'X_shadow_raw', 'X_test', 'X_test_raw', 'X_test_seq', 'X_train', 'X_train_raw', 'X_train_seq', 'feature_cols', 'scaler_mean', 'scaler_scale', 'seed', 'target_col', 'y', 'y_shadow', 'y_test', 'y_train']
